In [5]:
from pathlib import Path

import tensorflow as tf

from sklearn.model_selection import train_test_split

In [8]:
DATA_PATH = Path("../data/raw/seg_train")

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 42


image_paths = []
labels = []

class_names = sorted([
    folder.name
    for folder in DATA_PATH.iterdir()
    if folder.is_dir()
])

for label, class_name in enumerate(class_names):

    class_path = DATA_PATH / class_name

    for image_path in class_path.glob("*.jpg"):

        image_paths.append(
            str(image_path)
        )

        labels.append(
            label
        )

In [9]:
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels
)

In [10]:
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_labels
)

In [13]:
print(
    "Train:",
    len(train_paths)
)

print(
    "Validation:",
    len(val_paths)
)

print(
    "Test:",
    len(test_paths)
)

from collections import Counter

print(
    Counter(test_labels)
)

Train: 13627
Validation: 1703
Test: 1704
Counter({3: 304, 2: 296, 5: 289, 4: 278, 1: 274, 0: 263})


In [18]:
def load_image(path, label):

    image = tf.io.read_file(path)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    return image, label

In [19]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_paths,
        test_labels
    )
)

test_ds = test_ds.map(
    load_image
)

test_ds = test_ds.batch(
    BATCH_SIZE
)

In [20]:
model = tf.keras.models.load_model(
    "../models/best_model.keras",
)

54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.8680 - loss: 0.3719
Test accuracy: 0.8679577708244324
Test loss: 0.37192702293395996


              precision    recall  f1-score   support

   buildings       0.85      0.87      0.86       263
      forest       0.94      0.99      0.97       274
     glacier       0.89      0.72      0.79       296
    mountain       0.78      0.84      0.81       304
         sea       0.88      0.89      0.88       278
      street       0.88      0.91      0.89       289

    accuracy                           0.87      1704
   macro avg       0.87      0.87      0.87      1704
weighted avg       0.87      0.87      0.87      1704

